In [ ]:
 # Import libraries
import pandas as pd
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt



In [ ]:

image_dir = 'single_prediction'
image_data = []
filenames = []
images_to_display = []

for file in os.listdir(image_dir):
    if file.endswith('.jpg'):
        img_path = os.path.join(image_dir, file)
        try:
            image = Image.open(img_path).convert('L')
            image = image.resize((28, 28))
            images_to_display.append(image)
            pixels = np.array(image).flatten()
            image_data.append(pixels.tolist())
            filenames.append(file)
        except:
            print(f"Error reading {img_path}")

X_predict = torch.tensor(image_data, dtype=torch.float32)
predict_loader = DataLoader(X_predict, batch_size=1, shuffle=False)


In [ ]:

# Define model structure
class SimpleNN(nn.Module):
    def __init__(self, n_features=784, n_classes=2):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(n_features, 128),
            nn.ReLU(),
          
            nn.Linear(128, 64),
            nn.ReLU(),
          
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        return self.model(x)

# Load trained model
model = SimpleNN(n_features=784)
model.load_state_dict(torch.load('cats_vs_dogs_model.pth', map_location=torch.device('cpu')))
model.eval()

# Predict and show results
with torch.no_grad():
    for i, inputs in enumerate(predict_loader):
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        label = 'Cat' if predicted.item() == 0 else 'Dog'

        plt.imshow(images_to_display[i], cmap='gray')
        plt.title(f"Predicted: {label}")
        plt.axis('off')
        plt.show()

        print(f"{filenames[i]} --> {label}")
